# Módulo 04 · Embeddings

Un embedding representa el significado de un texto como un vector. Se comparan dos
implementaciones: LSA, escrita desde cero, y un modelo neuronal multilingüe.

In [1]:
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
DATA_DIR = ROOT / "data" / "novatech"

## Similitud coseno

Dos vectores que apuntan en la misma dirección tienen similitud 1; si son
perpendiculares, 0.

In [2]:
import numpy as np

from fundamenta.embeddings import cosine_similarity

a, b, c = np.array([1.0, 1.0]), np.array([2.0, 2.1]), np.array([1.0, -1.0])
print(f"a·b = {cosine_similarity(a, b):.3f}   (misma dirección)")
print(f"a·c = {cosine_similarity(a, c):.3f}   (perpendiculares)")

a·b = 1.000   (misma dirección)
a·c = 0.000   (perpendiculares)


## LSA: análisis semántico latente

Se construye la matriz TF-IDF de los fragmentos y se reduce con SVD. Los términos que
aparecen juntos quedan en direcciones próximas.

In [3]:
from fundamenta.chunking import chunk_documents, contextualize
from fundamenta.embeddings import LSAEmbedder
from fundamenta.loaders import load_directory

chunks = chunk_documents(load_directory(DATA_DIR))
texts = [contextualize(c) for c in chunks]

lsa = LSAEmbedder(dimensions=100).fit(texts)
lsa_vectors = lsa.embed_documents(texts)
print("Matriz de embeddings LSA:", lsa_vectors.shape)


def top(query_vector, vectors, k=3):
    scores = vectors @ query_vector
    for i in np.argsort(-scores)[:k]:
        print(
            f"  {scores[i]:.3f}  {chunks[i].source} | {chunks[i].text[:70].replace(chr(10), ' ')}…"
        )


query = "Me han robado el laptop, ¿qué tengo que hacer?"
print(f"\nConsulta: {query}")
top(lsa.embed_query(query), lsa_vectors)

Matriz de embeddings LSA: (41, 41)

Consulta: Me han robado el laptop, ¿qué tengo que hacer?
  0.000  beneficios-2026.pdf | Beneficios para empleados 2026 NovaTech (empresa ficticia) - Documento…
  0.000  beneficios-2026.pdf | 2. Ayuda para comida En España, cada día de trabajo presencial en la o…
  0.000  beneficios-2026.pdf | El presupuesto no utilizado no se acumula para el año siguiente. Las s…


Todas las similitudes son 0: ningún término de la consulta forma parte del
vocabulario aprendido, por lo que su vector es nulo. Además, la dimensión queda
limitada por el número de fragmentos (41). LSA necesita colecciones grandes para
aprender relaciones entre términos; con 41 fragmentos no puede saber que "laptop" y
"portátil" significan lo mismo.

## Modelo neuronal multilingüe

`intfloat/multilingual-e5-small` se ha entrenado con millones de pares de textos en
más de 90 idiomas. Se ejecuta en local (CPU) y los documentos no salen de la máquina.

In [4]:
from fundamenta.embeddings import SentenceTransformerEmbedder

e5 = SentenceTransformerEmbedder()
e5_vectors = e5.embed_documents(texts)
print("Matriz de embeddings E5:", e5_vectors.shape)
print(f"\nConsulta: {query}")
top(e5.embed_query(query), e5_vectors)

Matriz de embeddings E5: (41, 384)

Consulta: Me han robado el laptop, ¿qué tengo que hacer?
  0.886  faq-soporte-it.txt | ¿Qué hago si pierdo el portátil o me lo roban? Avisa a IT de inmediato…
  0.841  faq-soporte-it.txt | ¿Qué hago si olvido mi contraseña? Ve a login.novatech.example y pulsa…
  0.826  faq-soporte-it.txt | ¿Cómo me conecto a la VPN? Instala el cliente NovaConnect desde el Cen…


El modelo relaciona "robado el laptop" con "pierdo el portátil o me lo roban" sin
compartir apenas palabras.

## Similitud entre frases

In [5]:
sentences = [
    "Me han robado el ordenador del trabajo",
    "Si pierdes el portátil o te lo roban, avisa a IT",
    "La dieta diaria en España es de 45 euros",
    "¿Cuánto me pagan por comer fuera en un viaje?",
]
vectors = e5.embed_documents(sentences)
similarity = vectors @ vectors.T

print(" " * 6 + "".join(f"{j:>7}" for j in range(len(sentences))))
for i, row in enumerate(similarity):
    print(f"{i:>4}  " + "".join(f"{value:7.2f}" for value in row))
for i, sentence in enumerate(sentences):
    print(f"{i}: {sentence}")

            0      1      2      3
   0     1.00   0.89   0.85   0.84
   1     0.89   1.00   0.84   0.81
   2     0.85   0.84   1.00   0.86
   3     0.84   0.81   0.86   1.00
0: Me han robado el ordenador del trabajo
1: Si pierdes el portátil o te lo roban, avisa a IT
2: La dieta diaria en España es de 45 euros
3: ¿Cuánto me pagan por comer fuera en un viaje?


Las parejas con el mismo significado (0-1 y 2-3) obtienen las similitudes más altas.
En los modelos E5 los valores absolutos se concentran en un rango estrecho; lo que
importa es el orden relativo.